In [2]:
from pathlib import Path
import requests

def download_one_file_from_url(year: int, month: int) -> Path:
    """Downloads a single file from the specified URL and saves it to the local filesystem."""
    
    URL = f'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{year}-{month:02d}.parquet'
    response = requests.get(URL)
    
    if response.status_code == 200:
        path = f'../data/raw/rides_{year}-{month:02d}.parquet'
        open(path, 'wb').write(response.content)
        return path
    else:
        raise Exception(f'Failed to download file from {URL}. Status code: {response.status_code}')


In [3]:
download_one_file_from_url(year=2024, month=6)

'../data/raw/rides_2024-06.parquet'

In [11]:
import pandas as pd

rides = pd.read_parquet('../data/raw/rides_2024-06.parquet')
rides.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,1,2024-06-01 00:03:46,2024-06-01 00:31:23,1.0,12.50,1.0,N,138,195,1,48.5,7.75,0.5,11.55,0.0,1.0,69.30,0.0,1.75
1,2,2024-06-01 00:55:22,2024-06-01 01:08:24,1.0,4.34,1.0,N,138,7,1,20.5,6.00,0.5,8.40,0.0,1.0,38.15,0.0,1.75
2,1,2024-06-01 00:23:53,2024-06-01 00:32:35,1.0,1.30,1.0,N,166,41,1,10.0,1.00,0.5,3.10,0.0,1.0,15.60,0.0,0.00
3,1,2024-06-01 00:32:24,2024-06-01 00:40:06,1.0,1.20,1.0,N,148,114,1,8.6,3.50,0.5,0.20,0.0,1.0,13.80,2.5,0.00
4,1,2024-06-01 00:51:38,2024-06-01 00:58:17,1.0,1.00,1.0,N,148,249,1,7.2,3.50,0.5,2.00,0.0,1.0,14.20,2.5,0.00


In [12]:
rides = rides[['tpep_pickup_datetime', 'PULocationID']]

In [13]:
rides.rename(columns={
    'tpep_pickup_datetime': 'pickup_datetime',
    'PULocationID': 'pickup_location_id'
}, inplace=True)

rides.head(10)

,pickup_datetime,pickup_location_id
0,2024-06-01 00:03:46,138
1,2024-06-01 00:55:22,138
2,2024-06-01 00:23:53,166
3,2024-06-01 00:32:24,148
4,2024-06-01 00:51:38,148
5,2024-06-01 00:26:13,48
6,2024-06-01 00:01:04,132
7,2024-06-01 00:43:55,140
8,2024-05-31 23:38:07,230
9,2024-06-01 00:00:09,142


In [14]:
rides['pickup_datetime'].describe()

count                       3539193
mean     2024-06-15 19:50:24.269538
min             2008-12-31 00:00:00
25%             2024-06-08 00:37:52
50%             2024-06-15 14:26:05
75%             2024-06-23 10:29:20
max             2026-06-26 23:53:12
Name: pickup_datetime, dtype: object

In [15]:
rides = rides[rides.pickup_datetime >= '2024-06-01']
rides = rides[rides.pickup_datetime < '2024-07-01']
rides['pickup_datetime'].describe()


count                       3539142
mean     2024-06-15 19:59:13.480203
min             2024-06-01 00:00:00
25%             2024-06-08 00:38:09
50%             2024-06-15 14:26:18
75%             2024-06-23 10:29:26
max             2024-06-30 23:59:57
Name: pickup_datetime, dtype: object

In [16]:
rides.to_parquet('../data/transformed/validated_rides_2024_06.parquet')